# 05 — Topic-Level Sentiment Analysis
**Objective 5:** Measure sentiment *per topic* (staff, facilities, food, location, …) rather than only per review.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SENTIMENT_LABELS = ["negative", "neutral", "positive"]

def load_any(stem: Path):
    for s in (".parquet", ".pkl"):
        p = stem.with_suffix(s)
        if p.exists() and p.stat().st_size > 0:
            return pd.read_parquet(p) if s == ".parquet" else pd.read_pickle(p)
    return None

cleaned = load_any(DATA_DIR / "cleaned_reviews")
topics = load_any(MODELS_DIR / "document_topics")
assert cleaned is not None, "Run 02_data_cleaning first"
assert topics is not None, "Run 04_topic_modeling first"

merged = cleaned.merge(topics, on="review_id", how="inner")
print("Merged:", merged.shape)
merged.head(2)


In [ ]:
# Aggregate sentiment per topic
work = merged[merged["topic_id"] != -1].copy()

agg = (
    work.groupby(["topic_id", "topic_label"])
    .agg(
        n_docs=("sentiment_label", "size"),
        **{f"n_{s}": ("sentiment_label", lambda x, s=s: (x == s).sum()) for s in SENTIMENT_LABELS},
    )
    .reset_index()
)
for s in SENTIMENT_LABELS:
    agg[f"pct_{s}"] = (agg[f"n_{s}"] / agg["n_docs"] * 100).round(1)

pct_cols = [f"pct_{s}" for s in SENTIMENT_LABELS]
agg["dominant_sentiment"] = agg[pct_cols].idxmax(axis=1).str.replace("pct_", "")

if "rating_score" in work.columns:
    rating_agg = work.groupby("topic_id")["rating_score"].mean().round(2).rename("mean_rating")
    agg = agg.merge(rating_agg, left_on="topic_id", right_index=True, how="left")

topic_sent = agg[agg["n_docs"] >= 20].sort_values("n_docs", ascending=False).reset_index(drop=True)
display(topic_sent.head(15))


In [ ]:
# Stacked bar chart
plot_df = topic_sent.head(15)
fig, ax = plt.subplots(figsize=(12, 6))
bottom = None
colors = {"negative": "#e74c3c", "neutral": "#95a5a6", "positive": "#27ae60"}
for s in SENTIMENT_LABELS:
    vals = plot_df[f"pct_{s}"].values
    ax.barh(plot_df["topic_label"], vals, left=bottom, label=s, color=colors[s])
    bottom = vals if bottom is None else bottom + vals
ax.set_xlabel("% of Reviews")
ax.set_title("Sentiment Distribution by Topic (Top 15)")
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "10_topic_sentiment.png", dpi=150)
plt.show()


In [ ]:
# Priority topics (high negative share)
priority = topic_sent[
    (topic_sent["pct_negative"] >= 40)
    | (
        (topic_sent["dominant_sentiment"] == "negative")
        & (topic_sent["n_docs"] >= topic_sent["n_docs"].median())
    )
].sort_values(["pct_negative", "n_docs"], ascending=[False, False])

print("Priority (negative) topics:")
display(priority[["topic_label", "n_docs", "pct_negative", "pct_positive", "mean_rating"]].head(10)
        if "mean_rating" in priority.columns
        else priority[["topic_label", "n_docs", "pct_negative", "pct_positive"]].head(10))


In [ ]:
# Save topic-sentiment table
topic_sent.to_csv(MODELS_DIR / "topic_sentiment_table.csv", index=False)
print("Saved topic_sentiment_table.csv")
